# ChemiAI — Stacking (Meta-model)


## Что делаем
Строим stacking: базовые модели (RF, ExtraTrees, HistGBR) -> мета-модель Ridge по OOF предсказаниям.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge


In [ ]:
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
ID_COL = "index"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()


In [ ]:
# Мини-очистка признаков
const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(X_train)
Xte = imp.transform(X_test)


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_rf = np.zeros((len(Xtr), 3))
oof_et = np.zeros((len(Xtr), 3))
oof_hgb = np.zeros((len(Xtr), 3))
test_rf = np.zeros((len(Xte), 3))
test_et = np.zeros((len(Xte), 3))
test_hgb = np.zeros((len(Xte), 3))

for tr_idx, va_idx in kf.split(Xtr):
    X_tr, X_va = Xtr[tr_idx], Xtr[va_idx]
    y_tr = y_train.iloc[tr_idx].values

    for j in range(3):
        yj = np.log1p(y_tr[:, j])
        rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1, min_samples_leaf=2)
        et = ExtraTreesRegressor(n_estimators=700, random_state=42, n_jobs=-1, min_samples_leaf=2)
        hgb = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.03, max_iter=600, random_state=42)

        rf.fit(X_tr, yj); et.fit(X_tr, yj); hgb.fit(X_tr, yj)

        oof_rf[va_idx, j] = np.expm1(np.clip(rf.predict(X_va), 0, 12))
        oof_et[va_idx, j] = np.expm1(np.clip(et.predict(X_va), 0, 12))
        oof_hgb[va_idx, j] = np.expm1(np.clip(hgb.predict(X_va), 0, 12))

        test_rf[:, j] += np.expm1(np.clip(rf.predict(Xte), 0, 12)) / 5
        test_et[:, j] += np.expm1(np.clip(et.predict(Xte), 0, 12)) / 5
        test_hgb[:, j] += np.expm1(np.clip(hgb.predict(Xte), 0, 12)) / 5


In [ ]:
pred_test = np.zeros((len(Xte), 3))
for j in range(3):
    X_meta = np.column_stack([oof_rf[:, j], oof_et[:, j], oof_hgb[:, j]])
    X_meta_test = np.column_stack([test_rf[:, j], test_et[:, j], test_hgb[:, j]])

    meta = Ridge(alpha=3.0)
    meta.fit(X_meta, y_train.iloc[:, j].values)
    pred_test[:, j] = np.clip(meta.predict(X_meta_test), 0, None)

submission = sample_submission.copy()
submission["IC50"] = pred_test[:, 0]
submission["CC50"] = pred_test[:, 1]
submission["SI"] = pred_test[:, 2]
submission.to_csv("submission_stacking.csv", index=False)
submission.head()
